# Lab 3: What Should I Eat? (Auto-Graded)

You're going to build an agent that:
1. **Sees** a photo of your fridge or pantry
2. **Identifies** what ingredients are available
3. **Searches** the web for recipes using those ingredients
4. **Recommends** what you should make

**New this time: Your agent will be graded automatically.**

Run the grading cell. If you pass, you get 100. If not, you get 0.

Keep iterating until you pass.

---

## The Grading Rubric

Your agent will be tested with a photo of a fridge. To pass, it must:

| Requirement | Why |
|-------------|-----|
| Call `read_image` | The agent must actually **look** at the photo, not guess |
| Call `web_search` | The agent must actually **search** for recipes online |
| Call `web_fetch` | The agent must actually **read** a real recipe, not make one up |

**All three must happen.** If any is missing, you fail.

- ✅ **Pass = 100 points**
- ❌ **Fail = 0 points**

No partial credit. Your agent either does the work, or it doesn't.

---

## Setup

Run these cells to get started.

In [1]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://pypi.fury.io/ericmichael/

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

True

In [3]:
from agents import Agent
from omniagents import Runner
from omniagents.builtin.tools import web_search, web_fetch, read_image
from omniagents.notebook import evaluate

---

## Part 1: Understand the Tools

Before building the agent, understand what tools you have.

| Tool | What it does | Example use |
|------|--------------|-------------|
| `read_image` | Looks at an image and describes what it sees | Identify ingredients in a photo |
| `web_search` | Searches the web and returns results | Find recipes for "chicken and broccoli" |
| `web_fetch` | Fetches the full content of a webpage | Get the actual recipe from a URL |

**To pass the grading, your agent must use ALL THREE tools.**

---

## Part 2: Write the Instructions

The instructions tell the agent:
- What it is
- What tools it has
- **How to approach the task** (this is the key part!)

**Your job:** Fill in the `INSTRUCTIONS` below.

Think about:
- How do you tell the agent to **look at the photo first**?
- How do you tell it to **search the web** for recipes?
- How do you tell it to **fetch the actual recipe** from a URL?
- What should it return to the user?

In [4]:
from agents import function_tool


INSTRUCTIONS = """
You are a helpful cooking assistant.

When the user shows you a photo of their fridge or pantry:

1. First, use read_image to see what ingredients are available in their fridge, use read_image to see what seasoning they have, and see what meat they want to use.

2. Then go online and find recepipes that can be made with those ingredients using web_search and web_fetch.

3. Finally return a list of recipe suggestions to the user.

Always take into account the user's dietary preferences and restrictions: read the user's dietary preferences and restrictions from file using the read_user_preferences tool.

"""

@function_tool
def read_user_preferences() -> str:
    """Read the user's dietary preferences and restrictions from file."""
    try:
        with open("examples/lab2/user_preferences.json") as f:
            return f.read()
    except FileNotFoundError:
        return "No preferences file found."


---

## Part 3: Create the Agent

Create the agent with:
- A name
- Your instructions
- The tools it needs (remember: it must be ABLE to call `read_image`, `web_search`, AND `web_fetch`)
- A model (use `gpt-5.2`)

In [5]:
chef = Agent(
    name="chef",  # YOUR TURN: Give it a name
    instructions=INSTRUCTIONS,
    tools=[read_image, web_search, web_fetch, read_user_preferences],  # YOUR TURN: What tools does it need?
    model="gpt-5.2",
)

---

## Part 4: Test It!

Run your agent and watch what it does. Does it look at the photo? Search the web? Fetch a recipe?

![Sample Fridge](examples/lab3/fridge.jpg)

In [6]:
runner = Runner.from_agent(chef)
runner.run_notebook(input="What should I make for dinner? Here's what I have: examples/lab3/fridge.jpg")

---

## Part 5: Check Your Grade

Now run the auto-grader. **Green = pass. Red = fail.**

In [7]:
# === THE GRADER === #
# Do not modify this cell

result = await evaluate(
    chef,
    "What should I make for dinner? Here's what I have: examples/lab3/fridge.jpg",
    expect={
        "tool_called": ["read_image", "web_search", "web_fetch"],
    },
    max_turns=50,
)
result  # All three must be green to pass

Input,What should I make for dinner? Here's what I have: examples/lab3/fridge.jpg
Output,"From your fridge I can see: chicken (looks like thighs), a pack of sausages, BBQ sauce, lots of potatoes, a whole green cabbage, butter (Lurpak), and some salad/veg bits (plus assorted leftovers/condi..."
Tools Used,read_image → read_user_preferences → web_search → web_fetch → web_search → web_search → web_fetch → web_fetch
Measures,✓ tool_hallucination - all_tools_recognized
Expectations,✓ tool_called:read_image - read_image was called✓ tool_called:web_search - web_search was called✓ tool_called:web_fetch - web_fetch was called


---

## Part 6: Didn't Pass? Iterate!

Most people don't pass on the first try. That's normal.

**Green = pass. Red = fail.** The grader shows you exactly which tools were missing.

### Common Issues

**"read_image was not called"**
- Your instructions didn't tell the agent to look at the photo
- Try: "FIRST, use the read_image tool to see what ingredients are in the photo"

**"web_search was not called"**  
- Your instructions didn't tell the agent to search for recipes
- Try: "THEN, use web_search to find recipes that use those ingredients"

**"web_fetch was not called"**
- Your instructions didn't tell the agent to fetch the actual recipe
- Try: "NEXT, use web_fetch to get the full recipe from one of the URLs"

**Multiple tools missing?**
- Your instructions are probably too vague
- Be explicit about the workflow: first do X, then do Y, then do Z, finally return W

### The Fix

1. Go back to **Part 2**
2. Make your instructions more explicit
3. Re-run **Part 3** to recreate the agent
4. Re-run **Part 4** to test it
5. Re-run **Part 5** to check your grade
6. Repeat until it's green

---

## Challenge (Optional)

Already passing? Try this: change your model from `gpt-5.2` to `gpt-4.1` (a weaker model).

Can you get it to pass with the same instructions? Or do you need to be even more explicit?

---

## What You Just Learned

Without realizing it, you learned something important:

**You can test what an agent actually does, not just what it says.**

The grader didn't read the recipe and decide if it was good. It checked:
- Did the agent call `read_image`? (Did it look?)
- Did the agent call `web_search`? (Did it search?)
- Did the agent call `web_fetch`? (Did it read a real recipe?)

This is called **evaluation** — and it's how we make sure agents do what we want.

More on this in future notebooks.